In [74]:
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
from openai import OpenAI
import pandas as pd
import importlib
import helpers
import fitment_logic
import json

DEBUG=False

importlib.reload(helpers)
importlib.reload(fitment_logic)

load_dotenv()
print("Key found:", os.getenv("OPENAI_API_KEY") is not None)
api_key = os.getenv("OPENAI_API_KEY")
PROMPT_ID = os.getenv("PROMPT_ID")
PROMPT_VERSION = os.getenv("PROMPT_VERSION")

if not api_key:
    raise Exception("OPENAI_API_KEY not found")
if not PROMPT_ID:
    raise Exception("PROMPT_ID not found")
if not PROMPT_VERSION:
    raise Exception("PROMPT_VERSION not found")

client = OpenAI(api_key=api_key)

OUTPUT_FILE = "products_first_100.xlsx"
FITMENT_FILE = "nexus_ymms.csv"
CUSTOM_FITMENT_FILE = "custom_fitments.xlsx"

fitment_df = pd.read_csv(FITMENT_FILE)
fitment_df.columns = [col.strip().lower() for col in fitment_df.columns]
fitment_df["year"] = pd.to_numeric(fitment_df["year"], errors="coerce").astype("Int64")

if os.path.exists(CUSTOM_FITMENT_FILE):
    custom_fitment_df = pd.read_excel(CUSTOM_FITMENT_FILE)
else:
    custom_fitment_df = pd.DataFrame(columns=["SKU", "Year", "Make", "Model", "Submodel", "Universal"])

TITLE_COL = "Title"
TYPE_COL = "Type"
TAGS_COL = "Tags"
COLLECTIONS_COL = "Custom Collections"
WEIGHT_COL = "Variant Weight"
REVIEW_COL = "AI Review Reasons"

FITMENT_COL = "Metafield: convermax.fitment [list.single_line_text_field]"
UNIVERSAL_COL = "Metafield: convermax.universal [boolean]"

COMMAND_COL = "Command"
TAGS_COMMAND_COL = "Tags Command"
VARIANT_COMMAND_COL = "Variant Command"

signature_columns = [TITLE_COL, TYPE_COL, TAGS_COL, COLLECTIONS_COL, FITMENT_COL, UNIVERSAL_COL, COMMAND_COL, TAGS_COMMAND_COL, VARIANT_COMMAND_COL, REVIEW_COL]

print("FINISHED setup")

Key found: True
FINISHED setup


In [ ]:
df = pd.read_excel(OUTPUT_FILE)

START_ROW = 60
NUM_PRODUCTS_TO_PROCESS = 40
MAX_WORKERS = 10

for col in signature_columns:
    if col not in df.columns: df[col] = None
    df[col] = df[col].astype("object")

df[WEIGHT_COL] = pd.to_numeric(df[WEIGHT_COL], errors="coerce")

rows_to_process = list(df.iloc[START_ROW:START_ROW + NUM_PRODUCTS_TO_PROCESS].iterrows())

api_results = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [
        executor.submit(helpers.safe_call_ai_for_row, client, PROMPT_ID, PROMPT_VERSION, row_index, row)
        for row_index, row in rows_to_process
    ]

    for future in as_completed(futures):
        api_results.append(future.result())

for item in sorted(api_results, key=lambda x: x["row_index"]):
    row_index = item["row_index"]
    ai = item["ai_result"]
    original_row = df.loc[row_index]

    if ai.get("convermax_universal") is True:
        final_metadata = []
        fitment_tag = "fits_Universal"
        fitment_review_notes = []
        created_custom_fitments = []
    else:
        final_metadata, custom_fitment_df, fitment_review_notes, created_custom_fitments  = fitment_logic.resolve_vehicle_fitments_to_metadata(
            fitment_df=fitment_df,
            vehicle_fitments=ai.get("vehicle_fitments", []),
            custom_fitment_df=custom_fitment_df,
            custom_fitment_file=CUSTOM_FITMENT_FILE,
            product_row=original_row,
        )
        fitment_tag = fitment_logic.metadata_to_fitment_tag(final_metadata)
    if len(final_metadata) > 128:
        ai.setdefault("review_reasons", []).append(
            "fitment_metadata_exceeds_shopify_limit_128"
        )
    review_notes, ignored_review_notes = helpers.build_review_notes(ai, fitment_review_notes, final_metadata)
    helpers.print_product_summary(
        debug=DEBUG,
        row_index=row_index,
        original_row=original_row,
        ai=ai,
        final_metadata=final_metadata,
        fitment_tag=fitment_tag,
        review_notes=review_notes,
        ignored_review_notes=ignored_review_notes,
        created_custom_fitments=created_custom_fitments,
        title_col=TITLE_COL,
        type_col=TYPE_COL,
        weight_col=WEIGHT_COL,
    )
    helpers.update_product_row(
        df=df,
        row_index=row_index,
        original_row=original_row,
        ai=ai,
        review_notes=review_notes,
        fitment_tag=fitment_tag,
        final_metadata=final_metadata,
        review_col=REVIEW_COL,
        command_col=COMMAND_COL,
        tags_command_col=TAGS_COMMAND_COL,
        variant_command_col=VARIANT_COMMAND_COL,
        title_col=TITLE_COL,
        type_col=TYPE_COL,
        tags_col=TAGS_COL,
        collections_col=COLLECTIONS_COL,
        weight_col=WEIGHT_COL,
        fitment_col=FITMENT_COL,
        universal_col=UNIVERSAL_COL,
    )

df.to_excel(OUTPUT_FILE, index=False)
helpers.format_products_workbook(OUTPUT_FILE)

flagged_count = df[REVIEW_COL].notna().sum()
print(f"\nProducts requiring review: {flagged_count}")

print(f"\nDone. Saved updated file to: {OUTPUT_FILE}")

Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5.4-mini in organization org-Zf0JQcAgoidiYSyA6WVAiY96 on tokens per min (TPM): Limit 200000, Used 195242, Requested 10554. Please try again in 1.738s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}
Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5.4-mini in organization org-Zf0JQcAgoidiYSyA6WVAiY96 on tokens per min (TPM): Limit 200000, Used 200000, Requested 10614. Please try again in 3.184s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}
Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5.4-mini in organization org-Zf0JQcAgoidiYSyA6WVAiY96 on tokens per min (TPM): Limit 200000, Used 200000, Requested 10554. Please try again in 3.166s. Visit https://platform.openai.com/account/rate-limits to learn more.', 

In [77]:
ROW_INDEX = 26
MANUAL_FITMENTS = [
    # G11 / G12 — 7 Series

    {
        "start_year": 2016,
        "end_year": 2022,
        "make": "BMW",
        "model": "725d",
        "submodel": "",
    },
    {
        "start_year": 2016,
        "end_year": 2022,
        "make": "BMW",
        "model": "725Ld",
        "submodel": "",
    },
    {
        "start_year": 2016,
        "end_year": 2022,
        "make": "BMW",
        "model": "730d",
        "submodel": "",
    },
    {
        "start_year": 2016,
        "end_year": 2022,
        "make": "BMW",
        "model": "730Ld",
        "submodel": "",
    },
    {
        "start_year": 2016,
        "end_year": 2022,
        "make": "BMW",
        "model": "730i",
        "submodel": "",
    },
    {
        "start_year": 2016,
        "end_year": 2022,
        "make": "BMW",
        "model": "740d",
        "submodel": "",
    },
    {
        "start_year": 2016,
        "end_year": 2022,
        "make": "BMW",
        "model": "740Ld",
        "submodel": "",
    },
    {
        "start_year": 2016,
        "end_year": 2019,
        "make": "BMW",
        "model": "740e",
        "submodel": "",
    },
    {
        "start_year": 2016,
        "end_year": 2022,
        "make": "BMW",
        "model": "740i",
        "submodel": "",
    },
    {
        "start_year": 2016,
        "end_year": 2022,
        "make": "BMW",
        "model": "740Li",
        "submodel": "",
    },
    {
        "start_year": 2020,
        "end_year": 2022,
        "make": "BMW",
        "model": "745e",
        "submodel": "",
    },
    {
        "start_year": 2016,
        "end_year": 2020,
        "make": "BMW",
        "model": "750d",
        "submodel": "",
    },
    {
        "start_year": 2016,
        "end_year": 2020,
        "make": "BMW",
        "model": "750Ld",
        "submodel": "",
    },
    {
        "start_year": 2016,
        "end_year": 2022,
        "make": "BMW",
        "model": "750i",
        "submodel": "",
    },
    {
        "start_year": 2016,
        "end_year": 2022,
        "make": "BMW",
        "model": "750Li",
        "submodel": "",
    },
    {
        "start_year": 2017,
        "end_year": 2022,
        "make": "BMW",
        "model": "M760i",
        "submodel": "",
    },
    {
        "start_year": 2017,
        "end_year": 2022,
        "make": "BMW",
        "model": "M760Li",
        "submodel": "",
    },

    # G30 / G31 — 5 Series

    {
        "start_year": 2017,
        "end_year": 2023,
        "make": "BMW",
        "model": "530i",
        "submodel": "",
    },
    {
        "start_year": 2017,
        "end_year": 2023,
        "make": "BMW",
        "model": "540i",
        "submodel": "",
    },
    {
        "start_year": 2017,
        "end_year": 2020,
        "make": "BMW",
        "model": "M550d",
        "submodel": "",
    },
    {
        "start_year": 2018,
        "end_year": 2023,
        "make": "BMW",
        "model": "M550i",
        "submodel": "",
    },

    # G32 — 6 Series Gran Turismo

    {
        "start_year": 2018,
        "end_year": 2023,
        "make": "BMW",
        "model": "640i xDrive Gran Turismo",
        "submodel": "",
    },

    # G20 — 3 Series

    {
        "start_year": 2019,
        "end_year": 2026,
        "make": "BMW",
        "model": "M340i",
        "submodel": "",
    },

    # G14 / G15 / G16 — 8 Series

    {
        "start_year": 2019,
        "end_year": 2026,
        "make": "BMW",
        "model": "840d",
        "submodel": "",
    },
    {
        "start_year": 2019,
        "end_year": 2026,
        "make": "BMW",
        "model": "840i",
        "submodel": "",
    },
    {
        "start_year": 2020,
        "end_year": 2026,
        "make": "BMW",
        "model": "840i Gran Coupe",
        "submodel": "",
    },
    {
        "start_year": 2020,
        "end_year": 2026,
        "make": "BMW",
        "model": "840i xDrive Gran Coupe",
        "submodel": "",
    },
    {
        "start_year": 2019,
        "end_year": 2026,
        "make": "BMW",
        "model": "840i xDrive",
        "submodel": "",
    },

    # G01 / F97 — X3

    {
        "start_year": 2018,
        "end_year": 2024,
        "make": "BMW",
        "model": "X3",
        "submodel": "M40i",
    },
    {
        "start_year": 2018,
        "end_year": 2024,
        "make": "BMW",
        "model": "X3",
        "submodel": "xDrive30i",
    },
    {
        "start_year": 2019,
        "end_year": 2024,
        "make": "BMW",
        "model": "X3",
        "submodel": "sDrive30i",
    },
    {
        "start_year": 2020,
        "end_year": 2021,
        "make": "BMW",
        "model": "X3",
        "submodel": "xDrive30e",
    },
    {
        "start_year": 2020,
        "end_year": 2024,
        "make": "BMW",
        "model": "X3",
        "submodel": "M",
    },
    {
        "start_year": 2020,
        "end_year": 2024,
        "make": "BMW",
        "model": "X3",
        "submodel": "M Competition",
    },

    # G02 / F98 — X4

    {
        "start_year": 2019,
        "end_year": 2025,
        "make": "BMW",
        "model": "X4",
        "submodel": "M40i",
    },
    {
        "start_year": 2019,
        "end_year": 2025,
        "make": "BMW",
        "model": "X4",
        "submodel": "xDrive30i",
    },
    {
        "start_year": 2020,
        "end_year": 2025,
        "make": "BMW",
        "model": "X4",
        "submodel": "M",
    },
    {
        "start_year": 2020,
        "end_year": 2025,
        "make": "BMW",
        "model": "X4",
        "submodel": "M Competition",
    },

    # G05 / F95 — X5

    {
        "start_year": 2019,
        "end_year": 2020,
        "make": "BMW",
        "model": "X5",
        "submodel": "xDrive50i",
    },
    {
        "start_year": 2019,
        "end_year": 2026,
        "make": "BMW",
        "model": "X5",
        "submodel": "xDrive40i",
    },
    {
        "start_year": 2020,
        "end_year": 2026,
        "make": "BMW",
        "model": "X5",
        "submodel": "sDrive40i",
    },
    {
        "start_year": 2020,
        "end_year": 2026,
        "make": "BMW",
        "model": "X5",
        "submodel": "M",
    },
    {
        "start_year": 2020,
        "end_year": 2023,
        "make": "BMW",
        "model": "X5",
        "submodel": "M50i",
    },
    {
        "start_year": 2021,
        "end_year": 2023,
        "make": "BMW",
        "model": "X5",
        "submodel": "xDrive45e",
    },

    # G06 / F96 — X6

    {
        "start_year": 2020,
        "end_year": 2026,
        "make": "BMW",
        "model": "X6",
        "submodel": "xDrive40i",
    },
    {
        "start_year": 2020,
        "end_year": 2023,
        "make": "BMW",
        "model": "X6",
        "submodel": "M50i",
    },
    {
        "start_year": 2020,
        "end_year": 2026,
        "make": "BMW",
        "model": "X6",
        "submodel": "M",
    },
    {
        "start_year": 2020,
        "end_year": 2026,
        "make": "BMW",
        "model": "X6",
        "submodel": "M Competition",
    },

    # G07 — X7

    {
        "start_year": 2019,
        "end_year": 2020,
        "make": "BMW",
        "model": "X7",
        "submodel": "xDrive50i",
    },
    {
        "start_year": 2019,
        "end_year": 2026,
        "make": "BMW",
        "model": "X7",
        "submodel": "xDrive40i",
    },
    {
        "start_year": 2020,
        "end_year": 2022,
        "make": "BMW",
        "model": "X7",
        "submodel": "M50i",
    },

    # G29 — Z4

    {
        "start_year": 2019,
        "end_year": 2026,
        "make": "BMW",
        "model": "Z4",
        "submodel": "sDrive30i",
    },
    {
        "start_year": 2020,
        "end_year": 2026,
        "make": "BMW",
        "model": "Z4",
        "submodel": "sDrive M40i",
    },
]
AI_RESULT = {
  "title": "FI Exhaust Forged Carbon Paddle Shifters Diamond Black with Sporty Silver Color Stripes BMW G Series",
  "type": "Interior",
  "custom_collections": [
    "Carbon Accessories", "Interior Accessories"
  ],
  "vehicle_fitments": [

  ],
  "convermax_universal": False,
  "variant_weight": 1,
  "review_required": False,
  "review_reasons": []
}
(
    custom_fitment_df,
    final_metadata,
    fitment_tag,
    created_custom_fitments,
) = helpers.manually_resolve_fitments(
    df=df,
    fitment_df=fitment_df,
    custom_fitment_df=custom_fitment_df,
    custom_fitment_file=CUSTOM_FITMENT_FILE,
    row_index=ROW_INDEX,
    manual_fitments=MANUAL_FITMENTS,
    ai_result=AI_RESULT,
    review_col=REVIEW_COL,
    command_col=COMMAND_COL,
    tags_command_col=TAGS_COMMAND_COL,
    variant_command_col=VARIANT_COMMAND_COL,
    title_col=TITLE_COL,
    type_col=TYPE_COL,
    tags_col=TAGS_COL,
    collections_col=COLLECTIONS_COL,
    weight_col=WEIGHT_COL,
    fitment_col=FITMENT_COL,
    universal_col=UNIVERSAL_COL,
    debug=DEBUG,
)

df.to_excel(OUTPUT_FILE, index=False)
helpers.format_products_workbook(OUTPUT_FILE)


CUSTOM FITMENT CREATED FOR ROW: 26
SKU: TEMP-BMWGSFC-SS
AI RESPONSE:
{
  "title": "FI Exhaust Forged Carbon Paddle Shifters Diamond Black with Sporty Silver Color Stripes BMW G Series",
  "type": "Interior",
  "custom_collections": [
    "Carbon Accessories",
    "Interior Accessories"
  ],
  "vehicle_fitments": [],
  "convermax_universal": false,
  "variant_weight": 1,
  "review_required": false,
  "review_reasons": []
}
METADATA: ['2016-2019|BMW|740e|Base', '2016-2020|BMW|750Ld|Base', '2016-2020|BMW|750d|Base', '2016-2022|BMW|725Ld|Base', '2016-2022|BMW|725d|Base', '2016-2022|BMW|730Ld|Base', '2016-2022|BMW|730d|Base', '2016-2022|BMW|730i|Base', '2016-2022|BMW|740Ld|Base', '2016-2022|BMW|740Li|Base', '2016-2022|BMW|740d|Base', '2016-2022|BMW|740i|Base', '2016-2022|BMW|750Li|Base', '2016-2022|BMW|750i xDrive|Base', '2016-2022|BMW|750i|Base', '2017-2019|BMW|740e xDrive|iPerformance', '2017-2020|BMW|M550d|Base', '2017-2022|BMW|740i xDrive|Base', '2017-2022|BMW|M760Li|Base', '2017-2022|

In [76]:
df.to_excel(OUTPUT_FILE, index=False)

helpers.format_products_workbook(OUTPUT_FILE)

In [61]:
import pandas as pd

INPUT_FILE = "SignatureAutohausExport_2026-06-09_181952.xlsx"
OUTPUT_100_FILE = "products_first_100.xlsx"

df_100 = pd.read_excel(INPUT_FILE).head(100)

df_100.to_excel(OUTPUT_100_FILE, index=False)

print(f"Saved first 100 rows to {OUTPUT_100_FILE}")

Saved first 100 rows to products_first_100.xlsx


In [80]:
import pandas as pd

INPUT_FILE = "products_first_100.xlsx"
OUTPUT_FILE = "products_10_to_100_upload.xlsx"

df = pd.read_excel(INPUT_FILE)

# Keep product rows 10-100 (inclusive).
df_upload = df.iloc[9:100].copy()

# Remove AI Review Reasons column before upload.
if "AI Review Reasons" in df_upload.columns:
    df_upload = df_upload.drop(columns=["AI Review Reasons"])

df_upload.to_excel(OUTPUT_FILE, index=False)

print(f"Saved upload file: {OUTPUT_FILE}")
print(f"Rows in file excluding header: {len(df_upload)}")

Saved upload file: products_10_to_100_upload.xlsx
Rows in file excluding header: 91


In [ ]:
TEST_FILE = "SignatureAutohausExport_2026-06-09_181952.xlsx"
TEST_ROW_INDEX = 2

signature_df = pd.read_excel(TEST_FILE)

row = signature_df.loc[TEST_ROW_INDEX]
product = build_product_input(row)

print("INPUT PRODUCT:")
print(json.dumps(product, indent=2, ensure_ascii=False))

response = client.responses.create(
    prompt={
        "id": PROMPT_ID,
        "version": PROMPT_VERSION,
        "variables": {
            "product": json.dumps(product, ensure_ascii=False)
        }
    }
)

ai_result = json.loads(response.output_text)

print("\nAI RESPONSE:")
print(json.dumps(ai_result, indent=2, ensure_ascii=False))

if ai_result.get("convermax_universal") is True:
    final_metadata = []
    fitment_tag = ""
    lookup_notes = []
else:
    final_metadata, lookup_notes = resolve_vehicle_fitments_to_metadata(
        ai_result.get("vehicle_fitments", [])
    )
    fitment_tag = metadata_to_fitment_tag(final_metadata)

print("\nPYTHON FITMENT RESULT:")
print("METADATA:")
print(json.dumps(final_metadata, indent=2, ensure_ascii=False))

print("\nFITMENT TAG:")
print(fitment_tag)

print("\nLOOKUP NOTES:")
print(json.dumps(lookup_notes, indent=2, ensure_ascii=False))

INPUT PRODUCT:
{
  "title": "FI Exhaust Catback Valvetronic Muffler with Quad Tips BMW X6M 15-18",
  "body_html": "<div id=\"productDescription\">\n                            <p id=\"products_description\">Frequency Intelligent Valvetronic Exhaust System technology offers cutting-edge intelligent ECU exhaust control valve, with an emphasis on the optimization of both acoustics and performance. It is a testament to their philosophy of the ultimate union of comfort and performance experience for the driver and passengers.<br><br>When the valves are fully open for maximum flow and power, it creates an exotic tone and allows for high performance. When the valves are closed, volume is reduced for a more low-profile comfortable drive. With their latest technology, just one click on the remote control will setup any rpm to automatic mode. The automatic mode enables the system to detect the engine RPM to intelligently switch comfort/racing exhaust profiles. Users can also opt to simply switch

In [10]:
BMW_MODELS = [
    "316d",
    "318d",
    "318i",
    "320d",
    "320i",
    "325d",
    "330d",
    "330e",
    "330i",
    "335d",
    "335i",
    "340i",
    "420d",
    "420i",
    "430d",
    "430i",
    "435d",
    "440i",
    "520d",
    "520i",
    "525d",
    "528i",
    "530d",
    "530e",
    "530i",
    "535i",
    "540d",
    "540i",
    "550i",
    "630d",
    "630i",
    "640i",
    "650i",
    "725d",
    "725Ld",
    "730d",
    "730i",
    "730Ld",
    "740d",
    "740e",
    "740i",
    "740Ld",
    "740Li",
    "745e",
    "750d",
    "750i",
    "750Ld",
    "750Li",
    "760i",
    "760Li",
    "840d",
    "840i",
    "M340i",
    "M550d",
    "M550i",
    "M6",
    "M760i",
    "M760Li",
    "M8",
    "M850i",
    "X3",
    "X3 M",
    "X4",
    "X4 M",
    "X5",
    "X5 M",
    "X6",
    "X6 M",
    "X7",
    "Z4",
]

YEARS = range(2018, 2026)

found = []

missing = []

for model in BMW_MODELS:

    for year in YEARS:

        matches = fitment_df[

            (fitment_df["make"] == "BMW")

            & (fitment_df["model"] == model)

            & (fitment_df["year"] == year)

        ]

        if matches.empty:

            missing.append(f"{year}|BMW|{model}")

            continue

        for _, row in matches.iterrows():

            submodel = str(row["submodel"]).strip()

            metadata = (

                f"{year}|BMW|{model}"

                if not submodel

                else f"{year}|BMW|{model}|{submodel}"

            )

            found.append(metadata)

found = sorted(set(found))

missing = sorted(set(missing))

print("\nFOUND METADATA:\n")

for m in found:

    print(m)

print("\nMISSING FITMENTS:\n")

for m in missing:

    print(m)


FOUND METADATA:

2018|BMW|320i|Base
2018|BMW|330e|Base
2018|BMW|330i|Base
2018|BMW|340i|Base
2018|BMW|430i|Base
2018|BMW|440i|Base
2018|BMW|530e|Base
2018|BMW|530i|Base
2018|BMW|540i|Base
2018|BMW|640i|Base
2018|BMW|650i|Base
2018|BMW|740i|Base
2018|BMW|750i|Base
2018|BMW|M6|Base
2018|BMW|X3|M40i
2018|BMW|X3|xDrive30i
2018|BMW|X4|M40i
2018|BMW|X4|xDrive28i
2018|BMW|X5|M
2018|BMW|X5|sDrive35i
2018|BMW|X5|xDrive35d
2018|BMW|X5|xDrive35i
2018|BMW|X5|xDrive40e
2018|BMW|X5|xDrive50i
2018|BMW|X6|M
2018|BMW|X6|sDrive35i
2018|BMW|X6|xDrive35i
2018|BMW|X6|xDrive50i
2019|BMW|330i|Base
2019|BMW|430i|Base
2019|BMW|440i|Base
2019|BMW|530e|Base
2019|BMW|530i|Base
2019|BMW|540i|Base
2019|BMW|740i|Base
2019|BMW|750i|Base
2019|BMW|X3|M40i
2019|BMW|X3|sDrive30i
2019|BMW|X3|xDrive30i
2019|BMW|X4|M40i
2019|BMW|X4|xDrive30i
2019|BMW|X5|xDrive40i
2019|BMW|X5|xDrive50i
2019|BMW|X6|M
2019|BMW|X6|sDrive35i
2019|BMW|X6|xDrive35i
2019|BMW|X6|xDrive50i
2019|BMW|X7|xDrive40i
2019|BMW|X7|xDrive50i
2019|BMW|Z4|sDri

In [14]:
def metadata_to_fitment_tag(metadata_list):
    return "fits_" + "~".join(
        m.replace("|", "`") for m in metadata_list
    )

metadata = ["2015-2020|BMW|316d","2015-2020|BMW|318d","2015-2019|BMW|318i","2015-2021|BMW|320d","2018|BMW|320i|Base","2015-2021|BMW|325d","2015-2021|BMW|330d","2018|BMW|330e|Base","2021-2022|BMW|330e|Base","2018-2022|BMW|330i|Base","2015-2019|BMW|335d","2018|BMW|335i|Base","2018|BMW|340i|Base","2015-2020|BMW|420d","2015-2022|BMW|420i","2019-2020|BMW|430d","2018-2022|BMW|430i|Base","2019-2020|BMW|435d","2018-2020|BMW|440i|Base","2015-2020|BMW|520d","2015-2022|BMW|520i","2016-2020|BMW|525d","2017-2018|BMW|528i","2016-2020|BMW|530d","2018-2022|BMW|530e|Base","2018-2022|BMW|530i|Base","2017-2019|BMW|535i","2017-2020|BMW|540d","2018-2022|BMW|540i|Base","2017-2022|BMW|550i","2019-2022|BMW|640i","2018|BMW|640i|Base","2018|BMW|650i|Base","2019|BMW|725d","2019|BMW|725Ld","2019|BMW|730d","2016-2019|BMW|730i","2019|BMW|730Ld","2019|BMW|740d","2016-2019|BMW|740e","2018-2022|BMW|740i|Base","2019|BMW|740Ld","2016-2021|BMW|740Li","2017-2019|BMW|750d","2018-2019|BMW|750i|Base","2019|BMW|750Ld","2019|BMW|750Li","2019|BMW|760i","2019|BMW|760Li","2020-2022|BMW|840i|Base","2019-2022|BMW|M340i|Base","2017-2020|BMW|M550d","2018-2022|BMW|M550i xDrive|Base","2018|BMW|M6|Base","2019|BMW|M760i","2017-2019|BMW|M760Li","2020|BMW|M8|Base","2020|BMW|M8|Competition","2022|BMW|M8|Competition","2019-2022|BMW|M850i xDrive|Base","2020-2022|BMW|M850i xDrive Gran Coupe|Base","2020-2022|BMW|X3|M","2020-2022|BMW|X3|M Competition","2018-2022|BMW|X3|M40i","2019-2022|BMW|X3|sDrive30i","2020-2021|BMW|X3|xDrive30e","2018-2022|BMW|X3|xDrive30i","2020-2022|BMW|X4|M","2020-2022|BMW|X4|M Competition","2018-2022|BMW|X4|M40i","2018|BMW|X4|xDrive28i","2019-2022|BMW|X4|xDrive30i","2018-2022|BMW|X5|M","2020|BMW|X5|M Competition","2020-2022|BMW|X5|M50i","2018|BMW|X5|sDrive35i","2020-2022|BMW|X5|sDrive40i","2018|BMW|X5|xDrive35d","2018|BMW|X5|xDrive35i","2018|BMW|X5|xDrive40e","2019-2022|BMW|X5|xDrive40i","2021-2022|BMW|X5|xDrive45e","2018-2020|BMW|X5|xDrive50i","2018-2022|BMW|X6|M","2020-2021|BMW|X6|M Competition","2020-2022|BMW|X6|M50i","2018-2019|BMW|X6|sDrive35i","2020-2021|BMW|X6|sDrive40i","2018-2019|BMW|X6|xDrive35i","2020-2022|BMW|X6|xDrive40i","2018-2019|BMW|X6|xDrive50i","2020-2022|BMW|X7|M50i","2019-2022|BMW|X7|xDrive40i","2019-2020|BMW|X7|xDrive50i","2022|BMW|Z4|M40i","2020|BMW|Z4|M40i First Edition","2020-2021|BMW|Z4|sDrive M40i","2019-2022|BMW|Z4|sDrive30i"]

fitment_tag = metadata_to_fitment_tag(metadata)

print(fitment_tag)


fits_2015-2020`BMW`316d~2015-2020`BMW`318d~2015-2019`BMW`318i~2015-2021`BMW`320d~2018`BMW`320i`Base~2015-2021`BMW`325d~2015-2021`BMW`330d~2018`BMW`330e`Base~2021-2022`BMW`330e`Base~2018-2022`BMW`330i`Base~2015-2019`BMW`335d~2018`BMW`335i`Base~2018`BMW`340i`Base~2015-2020`BMW`420d~2015-2022`BMW`420i~2019-2020`BMW`430d~2018-2022`BMW`430i`Base~2019-2020`BMW`435d~2018-2020`BMW`440i`Base~2015-2020`BMW`520d~2015-2022`BMW`520i~2016-2020`BMW`525d~2017-2018`BMW`528i~2016-2020`BMW`530d~2018-2022`BMW`530e`Base~2018-2022`BMW`530i`Base~2017-2019`BMW`535i~2017-2020`BMW`540d~2018-2022`BMW`540i`Base~2017-2022`BMW`550i~2019-2022`BMW`640i~2018`BMW`640i`Base~2018`BMW`650i`Base~2019`BMW`725d~2019`BMW`725Ld~2019`BMW`730d~2016-2019`BMW`730i~2019`BMW`730Ld~2019`BMW`740d~2016-2019`BMW`740e~2018-2022`BMW`740i`Base~2019`BMW`740Ld~2016-2021`BMW`740Li~2017-2019`BMW`750d~2018-2019`BMW`750i`Base~2019`BMW`750Ld~2019`BMW`750Li~2019`BMW`760i~2019`BMW`760Li~2020-2022`BMW`840i`Base~2019-2022`BMW`M340i`Base~2017-2020`BMW

In [39]:
test_cases = [

    ("BMW", "M760Li", 2017, 2021),

    ("Audi", "RS4", 2017, 2025),

    ("Audi", "RS5", 2020, 2025),

    ("Audi", "R8", 2013, 2017),

]

for make, model, start_year, end_year in test_cases:

    matches = get_matching_csv_rows(

        fitment_df=fitment_df,

        start_year=start_year,

        end_year=end_year,

        make=make,

        model=model,

        submodel=""

    )

    print("=" * 80)

    print(f"{start_year}-{end_year} {make} {model}")

    print(f"Matches found: {len(matches)}")

    display(matches)

2017-2021 BMW M760Li
Matches found: 0


,year,make,model,submodel,url


2017-2025 Audi RS4
Matches found: 0


,year,make,model,submodel,url


2020-2025 Audi RS5
Matches found: 4


,year,make,model,submodel,url
3033,2021,Audi,RS5,Base,"AFE35-10027C,AFE77-92005,AFE77-92005-MC,AWE266..."
3034,2022,Audi,RS5,Base,"AFE35-10027C,AFE77-92005,AFE77-92005-MC,AWE266..."
3035,2023,Audi,RS5,Base,"AFE35-10027C,AFE77-92005,AFE77-92005-MC,AWE266..."
3036,2024,Audi,RS5,Base,"AFE35-10027C,AWE2660-15032,AWE3015-33123,AWE30..."


2013-2017 Audi R8
Matches found: 9


,year,make,model,submodel,url
2972,2014,Audi,R8,Base,"AFE10-10408DM,AFE10-10408RM,AFE57-10012R,AFE77..."
2973,2015,Audi,R8,Base,"AFE10-10408DM,AFE10-10408RM,AFE57-10012R,AFE77..."
2974,2017,Audi,R8,Base,"AFE57-10012R,AFE77-20008,AKRP-HF946,AKRTP-CT/3..."
2992,2014,Audi,R8,Plus,"AFE57-10012R,AFE77-16410,AFE77-20008,AKRP-HF88..."
2993,2015,Audi,R8,Plus,"AFE57-10012R,AFE77-16410,AFE77-20008,AKRP-HF88..."
2994,2017,Audi,R8,Plus,"AFE57-10012R,AFE77-20008,AKRP-HF946,AKRTP-CT/3..."
3000,2014,Audi,R8,Spyder,"AFE10-10408DM,AFE10-10408RM,AFE57-10012R,AFE77..."
3001,2015,Audi,R8,Spyder,"AFE10-10408DM,AFE10-10408RM,AFE57-10012R,AFE77..."
3002,2017,Audi,R8,Spyder,"AFE57-10012R,AFE77-20008,AKRP-HF946,AKRTP-CT/3..."


In [70]:
MANUAL_FITMENT_OVERRIDES = {
    ("Mercedes-Benz", "A250"): [
        {"start_year": 2012, "end_year": 2024, "metadata": "Mercedes-Benz|A250"},
    ],

    ("Mercedes-Benz", "E250"): [
        {"start_year": 2009, "end_year": 2013, "metadata": "Mercedes-Benz|E250"},
        {"start_year": 2014, "end_year": 2016, "metadata": "Mercedes-Benz|E250|Bluetec"},
        {"start_year": 2014, "end_year": 2016, "metadata": "Mercedes-Benz|E250|Bluetec 4Matic"},
        {"start_year": 2017, "end_year": 2020, "metadata": "Mercedes-Benz|E250"},
    ],
    ("Mercedes-Benz", "AMG GLC43"): [
        {"start_year": 2017, "end_year": 2026, "metadata": "Mercedes-Benz|GLC43 AMG|4Matic"},
    ],
    ("Audi", "S5 Sportback"): [
        {"start_year": 2008, "end_year": 2016, "metadata": "Audi|S5 Sportback"},
    ],
    ("Mercedes-Benz", "CLS350"): [
        {"start_year": 2010, "end_year": 2016, "metadata": "Mercedes-Benz|CLS350"},
    ],
    ("Mercedes-Benz", "CLS 350"): [
        {"start_year": 2010, "end_year": 2016, "metadata": "Mercedes-Benz|CLS350"},
    ],
    ("BMW", "M140i"): [
        {"start_year": 2016, "end_year": 2019, "metadata": "BMW|M140i"}
    ],
    ("BMW", "M340i"): [
        {"start_year": 2019, "end_year": 2025, "metadata": "BMW|M340i|Base"},
    ],
    ("Aston Martin", "V8 Vantage"): [
        {"start_year": 2018, "end_year": 2023, "metadata": "Aston Martin|Vantage|Base"},
        {"start_year": 2019, "end_year": 2019, "metadata": "Aston Martin|Vantage|AMR"},
        {"start_year": 2021, "end_year": 2023, "metadata": "Aston Martin|Vantage|F1 Edition"},
        {"start_year": 2023, "end_year": 2023, "metadata": "Aston Martin|Vantage|V12"},
    ],

    ("Aston Martin", "Vantage"): [
        {"start_year": 2018, "end_year": 2023, "metadata": "Aston Martin|Vantage|Base"},
        {"start_year": 2019, "end_year": 2019, "metadata": "Aston Martin|Vantage|AMR"},
        {"start_year": 2021, "end_year": 2023, "metadata": "Aston Martin|Vantage|F1 Edition"},
        {"start_year": 2023, "end_year": 2023, "metadata": "Aston Martin|Vantage|V12"},
    ],
    ("Mercedes-Benz", "SL500"): [
        {"start_year": 2007, "end_year": 2015, "metadata": "Mercedes-Benz|SL500"},
    ],
    ("Skoda", "SuperB"): [
        {"start_year": 2015, "end_year": 2015, "metadata": "Skoda|SuperB"},
    ],
    ("BMW", "M135i"): [
        {"start_year": 2019, "end_year": 2021, "metadata": "BMW|M135i"},
    ],
    ("Audi", "S3"): [
        {"start_year": 2003, "end_year": 2014, "metadata": "Audi|S3"},
    ],
    ("BMW", "220i"): [
    {"start_year": 2014, "end_year": 2024, "metadata": "BMW|220i"},
    ],
    ("BMW", "316d"): [
        {"start_year": 2008, "end_year": 2020, "metadata": "BMW|316d"},
    ],
    ("BMW", "316i"): [
        {"start_year": 2006, "end_year": 2018, "metadata": "BMW|316i"},
    ],
    ("BMW", "318d"): [
        {"start_year": 2006, "end_year": 2020, "metadata": "BMW|318d"},
    ],
    ("BMW", "318i"): [
        {"start_year": 2006, "end_year": 2019, "metadata": "BMW|318i"},
    ],
    ("BMW", "320d"): [
        {"start_year": 2006, "end_year": 2021, "metadata": "BMW|320d"},
    ],
    ("BMW", "325d"): [
        {"start_year": 2005, "end_year": 2021, "metadata": "BMW|325d"},
    ],
    ("BMW", "325i"): [
        {"start_year": 2007, "end_year": 2018, "metadata": "BMW|325i"},
    ],
    ("BMW", "330d"): [
        {"start_year": 2006, "end_year": 2021, "metadata": "BMW|330d"},
    ],
    ("BMW", "335d"): [
        {"start_year": 2006, "end_year": 2019, "metadata": "BMW|335d"},
    ],
    ("BMW", "335is"): [
        {"start_year": 2012, "end_year": 2018, "metadata": "BMW|335is"},
    ],
    ("BMW", "420d"): [
        {"start_year": 2015, "end_year": 2020, "metadata": "BMW|420d"},
    ],
    ("BMW", "420i"): [
        {"start_year": 2014, "end_year": 2023, "metadata": "BMW|420i"},
    ],
    ("Ferrari", "GTC4Lusso"): [
        {"start_year": 2016, "end_year": 2016, "metadata": "Ferrari|GTC4Lusso"},
    ],
    ("BMW", "430d"): [{"start_year": 2019, "end_year": 2020, "metadata": "BMW|430d"}],
("BMW", "435d"): [{"start_year": 2019, "end_year": 2020, "metadata": "BMW|435d"}],
("BMW", "520d"): [{"start_year": 2011, "end_year": 2020, "metadata": "BMW|520d"}],
("BMW", "520i"): [{"start_year": 2011, "end_year": 2022, "metadata": "BMW|520i"}],
("BMW", "525d"): [{"start_year": 2016, "end_year": 2020, "metadata": "BMW|525d"}],
("BMW", "528i"): [{"start_year": 2017, "end_year": 2018, "metadata": "BMW|528i"}],
("BMW", "530d"): [{"start_year": 2016, "end_year": 2020, "metadata": "BMW|530d"}],
("BMW", "535i"): [{"start_year": 2017, "end_year": 2019, "metadata": "BMW|535i"}],
("BMW", "540d"): [{"start_year": 2017, "end_year": 2020, "metadata": "BMW|540d"}],
("BMW", "550i"): [{"start_year": 2017, "end_year": 2022, "metadata": "BMW|550i"}],
("BMW", "640i"): [{"start_year": 2019, "end_year": 2023, "metadata": "BMW|640i"}],
("BMW", "650i"): [{"start_year": 2011, "end_year": 2011, "metadata": "BMW|650i"}],
("BMW", "725d"): [{"start_year": 2019, "end_year": 2019, "metadata": "BMW|725d"}],
("BMW", "725Ld"): [{"start_year": 2019, "end_year": 2019, "metadata": "BMW|725Ld"}],
("BMW", "730d"): [{"start_year": 2019, "end_year": 2019, "metadata": "BMW|730d"}],
("BMW", "730i"): [{"start_year": 2016, "end_year": 2019, "metadata": "BMW|730i"}],
("BMW", "730Ld"): [{"start_year": 2019, "end_year": 2019, "metadata": "BMW|730Ld"}],
("BMW", "740d"): [{"start_year": 2019, "end_year": 2019, "metadata": "BMW|740d"}],
("BMW", "740e"): [{"start_year": 2016, "end_year": 2019, "metadata": "BMW|740e"}],
("BMW", "740Ld"): [{"start_year": 2019, "end_year": 2019, "metadata": "BMW|740Ld"}],
("BMW", "740Li"): [{"start_year": 2016, "end_year": 2021, "metadata": "BMW|740Li"}],
("BMW", "750d"): [{"start_year": 2017, "end_year": 2019, "metadata": "BMW|750d"}],
("BMW", "750Ld"): [{"start_year": 2019, "end_year": 2019, "metadata": "BMW|750Ld"}],
("BMW", "750Li"): [{"start_year": 2019, "end_year": 2019, "metadata": "BMW|750Li"}],
("BMW", "760i"): [{"start_year": 2019, "end_year": 2019, "metadata": "BMW|760i"}],
("BMW", "760Li"): [{"start_year": 2019, "end_year": 2019, "metadata": "BMW|760Li"}],
("BMW", "M550d"): [{"start_year": 2017, "end_year": 2020, "metadata": "BMW|M550d"}],
("BMW", "M760i"): [{"start_year": 2019, "end_year": 2019, "metadata": "BMW|M760i"}],
("BMW", "M760Li"): [{"start_year": 2017, "end_year": 2019, "metadata": "BMW|M760Li"}],
("Audi", "RS4"): [{"start_year": 2017, "end_year": 2023, "metadata": "Audi|RS4"},],
("Audi", "RS5"): [{"start_year": 2020, "end_year": 2020, "metadata": "Audi|RS5"},],
("Audi", "R8"): [{"start_year": 2013, "end_year": 2013, "metadata": "Audi|R8"},],

}

In [81]:
import ast
import json
from pathlib import Path

import pandas as pd


INPUT_FILE = "Import_Result_2026-07-18_152703.xlsx"
OUTPUT_FILE = "Import_Result_2026-07-18_152703_failed_retry.xlsx"

IMPORT_RESULT_COLUMN = "Import Result"
IMPORT_COMMENT_COLUMN = "Import Comment"
TAGS_COLUMN = "Tags"

FITMENT_METAFIELD_COLUMN = (
    "Metafield: convermax.fitment [list.single_line_text_field]"
)


def metadata_to_fitment_tag(
    metadata_rows,
    max_tag_length=255,
):
    """
    Converts Convermax metadata rows into one or more comma-separated
    Shopify product tags.

    Example input:
        [
            "2019-2021|BMW|M340i|Base",
            "2020-2022|BMW|M340i xDrive|Base",
        ]

    Example output:
        fits_2019-2021`BMW`M340i`Base,
        fits_2020-2022`BMW`M340i xDrive`Base

    Each individual Shopify tag is no longer than 255 characters.
    """

    if not metadata_rows:
        return ""

    converted_rows = []

    for value in metadata_rows:
        if pd.isna(value):
            continue

        row = str(value).strip()

        if not row:
            continue

        converted_rows.append(row.replace("|", "`"))

    if not converted_rows:
        return ""

    fitment_tags = []
    current_rows = []

    for row in converted_rows:
        single_tag = f"fits_{row}"

        if len(single_tag) > max_tag_length:
            raise ValueError(
                "An individual fitment row is too long for Shopify: "
                f"{len(single_tag)} characters: {single_tag}"
            )

        candidate = "fits_" + "~".join(current_rows + [row])

        if len(candidate) <= max_tag_length:
            current_rows.append(row)
        else:
            fitment_tags.append(
                "fits_" + "~".join(current_rows)
            )
            current_rows = [row]

    if current_rows:
        fitment_tags.append(
            "fits_" + "~".join(current_rows)
        )

    # The commas make these separate Shopify tags.
    return ", ".join(fitment_tags)


def parse_fitment_metadata(value):
    """
    Parses the fitment metafield value from Excel.

    Supports:
    - Actual Python lists
    - JSON arrays
    - Python-list strings
    - Newline-separated metadata
    - Comma-separated metadata
    - A single metadata row
    """

    if value is None or pd.isna(value):
        return []

    if isinstance(value, (list, tuple, set)):
        return [
            str(item).strip()
            for item in value
            if str(item).strip()
        ]

    text = str(value).strip()

    if not text:
        return []

    # First try JSON, for example:
    # ["2019-2021|BMW|M340i|Base"]
    try:
        parsed = json.loads(text)

        if isinstance(parsed, list):
            return [
                str(item).strip()
                for item in parsed
                if str(item).strip()
            ]
    except (json.JSONDecodeError, TypeError):
        pass

    # Then try a Python list representation, for example:
    # ['2019-2021|BMW|M340i|Base']
    try:
        parsed = ast.literal_eval(text)

        if isinstance(parsed, (list, tuple, set)):
            return [
                str(item).strip()
                for item in parsed
                if str(item).strip()
            ]
    except (ValueError, SyntaxError):
        pass

    # Support newline-separated values.
    if "\n" in text:
        return [
            item.strip().strip('"').strip("'")
            for item in text.splitlines()
            if item.strip()
        ]

    # Support comma-separated values where each item is metadata.
    if "," in text:
        comma_parts = [
            item.strip().strip('"').strip("'")
            for item in text.split(",")
            if item.strip()
        ]

        if all("|" in item for item in comma_parts):
            return comma_parts

    # Otherwise treat the entire value as one metadata row.
    return [text.strip('"').strip("'")]


def split_shopify_tags(value):
    """
    Converts a comma-separated Shopify Tags cell into a clean list.
    """

    if value is None or pd.isna(value):
        return []

    return [
        tag.strip()
        for tag in str(value).split(",")
        if tag.strip()
    ]


def replace_fitment_tags(current_tags, new_fitment_tag_string):
    """
    Removes every existing tag beginning with fits_, case-insensitively,
    then appends the newly generated fitment tags.
    """

    tags = split_shopify_tags(current_tags)

    regular_tags = [
        tag
        for tag in tags
        if not tag.lower().startswith("fits_")
    ]

    new_fitment_tags = split_shopify_tags(
        new_fitment_tag_string
    )

    updated_tags = regular_tags + new_fitment_tags

    # Preserve order while removing exact duplicate tags.
    unique_tags = []
    seen = set()

    for tag in updated_tags:
        normalized = tag.casefold()

        if normalized not in seen:
            unique_tags.append(tag)
            seen.add(normalized)

    return ", ".join(unique_tags)


def find_column_case_insensitive(df, requested_name):
    """
    Finds a dataframe column without requiring exact capitalization
    or exact surrounding whitespace.
    """

    normalized_requested = requested_name.strip().casefold()

    for column in df.columns:
        if str(column).strip().casefold() == normalized_requested:
            return column

    raise KeyError(
        f"Column not found: {requested_name}\n"
        f"Available columns:\n{list(df.columns)}"
    )


input_path = Path(INPUT_FILE)

if not input_path.exists():
    raise FileNotFoundError(
        f"Input file does not exist: {input_path.resolve()}"
    )

df = pd.read_excel(input_path)

import_result_col = find_column_case_insensitive(
    df,
    IMPORT_RESULT_COLUMN,
)
import_comment_col = find_column_case_insensitive(
    df,
    IMPORT_COMMENT_COLUMN,
)
tags_col = find_column_case_insensitive(
    df,
    TAGS_COLUMN,
)
fitment_metafield_col = find_column_case_insensitive(
    df,
    FITMENT_METAFIELD_COLUMN,
)


# Copy every failed row into the retry workbook.
failed_mask = (
    df[import_result_col]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.casefold()
    .eq("failed")
)

df_retry = df.loc[failed_mask].copy()


updated_tag_rows = 0
metadata_parse_failures = []


for row_index, row in df_retry.iterrows():
    import_comment = str(
        row.get(import_comment_col, "")
    ).strip()

    # A comment can sometimes contain more text than only the error.
    # Therefore, use a case-insensitive containment check.
    is_invalid_tags_error = (
        "product tags is invalid"
        in import_comment.casefold()
    )

    if not is_invalid_tags_error:
        # Keep the failed row unchanged for later manual correction.
        continue

    raw_metadata = row.get(fitment_metafield_col)

    try:
        metadata_rows = parse_fitment_metadata(
            raw_metadata
        )

        if not metadata_rows:
            metadata_parse_failures.append(
                {
                    "row_index": row_index,
                    "reason": "Fitment metadata is empty",
                    "value": raw_metadata,
                }
            )
            continue

        recreated_fitment_tags = metadata_to_fitment_tag(
            metadata_rows
        )

        updated_tags = replace_fitment_tags(
            current_tags=row.get(tags_col),
            new_fitment_tag_string=recreated_fitment_tags,
        )

        df_retry.at[row_index, tags_col] = updated_tags
        updated_tag_rows += 1

    except Exception as exc:
        # Keep the row in the output even when its metadata cannot
        # currently be parsed or converted.
        metadata_parse_failures.append(
            {
                "row_index": row_index,
                "reason": str(exc),
                "value": raw_metadata,
            }
        )


df_retry.to_excel(
    OUTPUT_FILE,
    index=False,
)


print(f"Input rows: {len(df)}")
print(f"Failed rows copied: {len(df_retry)}")
print(f"Rows with rebuilt fitment tags: {updated_tag_rows}")
print(
    "Failed rows left unchanged for manual review: "
    f"{len(df_retry) - updated_tag_rows}"
)
print(f"Saved retry file: {OUTPUT_FILE}")


if metadata_parse_failures:
    print("\nMetadata values that could not be rebuilt:")

    for failure in metadata_parse_failures:
        print("-" * 80)
        print(f"Excel dataframe index: {failure['row_index']}")
        print(f"Reason: {failure['reason']}")
        print(f"Value: {failure['value']}")

Input rows: 91
Failed rows copied: 31
Rows with rebuilt fitment tags: 24
Failed rows left unchanged for manual review: 7
Saved retry file: Import_Result_2026-07-18_152703_failed_retry.xlsx


In [82]:
from pathlib import Path

import pandas as pd


INPUT_FILE = (
    "Import_Result_2026-07-18_152703_failed_retry.xlsx"
)

OUTPUT_FILE = (
    "Import_Result_2026-07-18_152703_failed_retry_fixed.xlsx"
)

FITMENT_COLUMN_NAME = (
    "Metafield: convermax.fitment "
    "[list.single_line_text_field]"
)


def find_column_case_insensitive(
    df: pd.DataFrame,
    requested_name: str,
) -> str:
    """
    Finds a column while ignoring capitalization and surrounding spaces.
    """

    normalized_requested = requested_name.strip().casefold()

    for column in df.columns:
        normalized_column = str(column).strip().casefold()

        if normalized_column == normalized_requested:
            return column

    raise KeyError(
        f"Column not found: {requested_name}\n"
        f"Available columns:\n{list(df.columns)}"
    )


def is_universal_metadata(value) -> bool:
    """
    Returns True for representations of the invalid universal metadata.

    Examples recognized:
        ["Universal"]
        ['Universal']
        [ "Universal" ]
    """

    if value is None or pd.isna(value):
        return False

    normalized = (
        str(value)
        .strip()
        .replace(" ", "")
        .replace("'", '"')
        .casefold()
    )

    return normalized == '["universal"]'


input_path = Path(INPUT_FILE)

if not input_path.exists():
    raise FileNotFoundError(
        f"Input file not found: {input_path.resolve()}"
    )


df = pd.read_excel(input_path)

fitment_column = find_column_case_insensitive(
    df,
    FITMENT_COLUMN_NAME,
)


# Find rows containing ["Universal"] in the fitment metafield.
universal_metadata_mask = df[fitment_column].apply(
    is_universal_metadata
)

replacement_count = int(universal_metadata_mask.sum())


# Blank the fitment metafield.
#
# We intentionally use an empty string instead of the literal text "[]".
# The resulting Excel cell is blank, while the universal Boolean and
# fits_Universal tag remain unchanged.
df.loc[universal_metadata_mask, fitment_column] = ""


# Remove Matrixify result columns if they exist.
columns_to_remove = []

for requested_column in [
    "Import Result",
    "Import Comment",
]:
    try:
        actual_column = find_column_case_insensitive(
            df,
            requested_column,
        )
        columns_to_remove.append(actual_column)

    except KeyError:
        print(
            f"Column was not present and was skipped: "
            f"{requested_column}"
        )


if columns_to_remove:
    df = df.drop(columns=columns_to_remove)


df.to_excel(
    OUTPUT_FILE,
    index=False,
)


print(f"Input file: {INPUT_FILE}")
print(f'Values changed from ["Universal"] to blank: {replacement_count}')
print(f"Columns removed: {columns_to_remove}")
print(f"Rows in output file: {len(df)}")
print(f"Saved upload file: {OUTPUT_FILE}")

Input file: Import_Result_2026-07-18_152703_failed_retry.xlsx
Values changed from ["Universal"] to blank: 7
Columns removed: ['Import Result', 'Import Comment']
Rows in output file: 31
Saved upload file: Import_Result_2026-07-18_152703_failed_retry_fixed.xlsx


In [83]:
import pandas as pd

INPUT_FILE = "Import_Result_2026-07-20_163415.xlsx"
OUTPUT_FILE = "Import_Result_2026-07-20_163415_128_retry.xlsx"


def find_column(df, column_name):
    """
    Finds a column ignoring capitalization and surrounding whitespace.
    """
    normalized = column_name.strip().casefold()

    for column in df.columns:
        if str(column).strip().casefold() == normalized:
            return column

    raise KeyError(f"Column not found: {column_name}")


df = pd.read_excel(INPUT_FILE)

import_result_col = find_column(df, "Import Result")
import_comment_col = find_column(df, "Import Comment")

# Keep only rows that failed because of the 128 element limit.
mask = (
    df[import_result_col]
    .fillna("")
    .astype(str)
    .str.casefold()
    .eq("failed")
) & (
    df[import_comment_col]
    .fillna("")
    .astype(str)
    .str.contains(
        "128 elements",
        case=False,
        regex=False,
    )
)

df_retry = df.loc[mask].copy()

# Remove Matrixify import columns.
df_retry.drop(
    columns=[
        import_result_col,
        import_comment_col,
    ],
    inplace=True,
)

df_retry.to_excel(
    OUTPUT_FILE,
    index=False,
)

print(f"Rows kept: {len(df_retry)}")
print(f"Saved: {OUTPUT_FILE}")

Rows kept: 1
Saved: Import_Result_2026-07-20_163415_128_retry.xlsx


In [84]:
list_m = ["2016-2017|BMW|535i GT xDrive|Base", "2016-2017|BMW|535i GT|Base", "2016-2017|BMW|550i GT xDrive|Base", "2016-2018|BMW|320i xDrive|Base", "2016-2018|BMW|340i xDrive|Base", "2016-2018|BMW|640i xDrive|Base", "2016-2018|BMW|650i xDrive|Base", "2016-2019|BMW|640i Gran Coupe|Base", "2016-2019|BMW|640i xDrive Gran Coupe|Base", "2016-2019|BMW|650i Gran Coupe|Base", "2016-2019|BMW|650i xDrive Gran Coupe|Base", "2016-2020|BMW|316d|Base", "2016-2020|BMW|318d|Base", "2016-2020|BMW|318i|Base", "2016-2020|BMW|320d|Base", "2016-2020|BMW|320i|Base", "2016-2020|BMW|325d|Base", "2016-2020|BMW|330d|Base", "2016-2020|BMW|330i|Base", "2016-2020|BMW|335d|Base", "2016-2020|BMW|335i|Base", "2016-2020|BMW|340i|Base", "2016-2020|BMW|420d|Base", "2016-2020|BMW|420i|Base", "2016-2020|BMW|430d|Base", "2016-2020|BMW|430i|Base", "2016-2020|BMW|435d|Base", "2016-2020|BMW|440i|Base", "2016-2020|BMW|520d|Base", "2016-2020|BMW|520i|Base", "2016-2020|BMW|525d|Base", "2016-2020|BMW|528i|Base", "2016-2020|BMW|530d|Base", "2016-2020|BMW|530e|Base", "2016-2020|BMW|530i|Base", "2016-2020|BMW|535i|Base", "2016-2020|BMW|540d|Base", "2016-2020|BMW|540i|Base", "2016-2020|BMW|550i|Base", "2016-2020|BMW|630d|Base", "2016-2020|BMW|630i|Base", "2016-2020|BMW|640i|Base", "2016-2020|BMW|650i|Base", "2016-2020|BMW|725Ld|Base", "2016-2020|BMW|725d|Base", "2016-2020|BMW|730Ld|Base", "2016-2020|BMW|730d|Base", "2016-2020|BMW|730i|Base", "2016-2020|BMW|740Ld|Base", "2016-2020|BMW|740Li|Base", "2016-2020|BMW|740d|Base", "2016-2020|BMW|740e|Base", "2016-2020|BMW|740i|Base", "2016-2020|BMW|745e|Base", "2016-2020|BMW|750Ld|Base", "2016-2020|BMW|750Li|Base", "2016-2020|BMW|750d|Base", "2016-2020|BMW|750i xDrive|Base", "2016-2020|BMW|750i|Base", "2016-2020|BMW|760Li|Base", "2016-2020|BMW|760i|Base", "2016|BMW|330e|Base", "2016|BMW|335i GT xDrive|Base", "2016|BMW|528i xDrive|Base", "2016|BMW|535i xDrive|Base", "2016|BMW|550i xDrive|Base", "2017-2019|BMW|330i GT xDrive|Base", "2017-2019|BMW|340i GT xDrive|Base", "2017-2019|BMW|740e xDrive|iPerformance", "2017-2020|BMW|330i xDrive|Base", "2017-2020|BMW|430i Gran Coupe|Base", "2017-2020|BMW|430i xDrive Gran Coupe|Base", "2017-2020|BMW|430i xDrive|Base", "2017-2020|BMW|440i Gran Coupe|Base", "2017-2020|BMW|440i xDrive Gran Coupe|Base", "2017-2020|BMW|440i xDrive|Base", "2017-2020|BMW|530i xDrive|Base", "2017-2020|BMW|540i xDrive|Base", "2017-2020|BMW|740i xDrive|Base", "2017-2021|BMW|M550d|Base", "2017-2021|BMW|M550i|Base", "2017|BMW|330e|iPerformance", "2018-2019|BMW|640i xDrive Gran Turismo|Base", "2018-2020|BMW|330e|Base", "2018-2020|BMW|530e xDrive|Base", "2018-2021|BMW|M550i xDrive|Base", "2018-2022|BMW|X3|M", "2018-2022|BMW|X3|M40i", "2018-2022|BMW|X3|xDrive30i", "2018|BMW|540d xDrive|Base", "2019-2020|BMW|330e|iPerformance", "2019-2020|BMW|X5|xDrive50i", "2019-2020|BMW|X7|xDrive50i", "2019-2022|BMW|840d|Base", "2019-2022|BMW|840i|Base", "2019-2022|BMW|M340i|Base", "2019-2022|BMW|M760Li|Base", "2019-2022|BMW|M760i xDrive|Base", "2019-2022|BMW|M760i|Base", "2019-2022|BMW|X3|sDrive30i", "2019-2022|BMW|X4|M", "2019-2022|BMW|X4|M40i", "2019-2022|BMW|X4|xDrive30i", "2019-2022|BMW|X5|M", "2019-2022|BMW|X5|xDrive40i", "2019-2022|BMW|X7|xDrive40i", "2019-2022|BMW|Z4|sDrive30i", "2020-2021|BMW|X3|xDrive30e", "2020-2021|BMW|X6|M Competition", "2020-2021|BMW|X6|sDrive40i", "2020-2021|BMW|Z4|sDrive M40i", "2020-2022|BMW|840i Gran Coupe|Base", "2020-2022|BMW|840i xDrive Gran Coupe|Base", "2020-2022|BMW|840i xDrive|Base", "2020-2022|BMW|M340i xDrive|Base", "2020-2022|BMW|X3|M", "2020-2022|BMW|X3|M Competition", "2020-2022|BMW|X4|M", "2020-2022|BMW|X4|M Competition", "2020-2022|BMW|X5|M", "2020-2022|BMW|X5|M50i", "2020-2022|BMW|X5|sDrive40i", "2020-2022|BMW|X6|M", "2020-2022|BMW|X6|M50i", "2020-2022|BMW|X6|xDrive40i", "2020-2022|BMW|X7|M50i", "2020|BMW|745e xDrive|Base", "2020|BMW|X5|M Competition", "2020|BMW|Z4|M40i First Edition", "2021-2022|BMW|X5|xDrive45e", "2022|BMW|Z4|M40i"]
print(len(list_m))

131
